# A1 corregido: Regresión logística (correcciones i y iii)

Cambios respecto a `A1_RegresionLogistica.ipynb` (el original queda intacto):
1. La etiqueta es binaria, `was_preterm` (0 = término, 1 = pretérmino), igual que en A2 y A3.
   Antes eran 3 clases (`term`, `preterm`, `early_preterm`).
2. El `train_test_split` por muestra se reemplaza por los pliegues comunes por paciente
   (`pliegues_por_paciente.csv`, 5 pliegues x 5 repeticiones).
3. La probabilidad de cada paciente es el promedio de las de sus muestras, y se evalúa por paciente.
4. (Punto iii) El umbral de decisión es el índice de Youden, max(TPR − FPR), calculado solo con las
   pacientes de entrenamiento de cada pliegue mediante una validación cruzada interna de 3 pliegues
   por paciente (`comun.umbral_interno`). El pliegue de prueba no interviene en el umbral.

Lo que ve el modelo (solo la categoría racial codificada como entero) **no** cambia aquí: es del punto iv.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, os.path.abspath('.'))
from comun import (leer_datos, RESULTADOS, ETIQUETA, cargar_pliegues, iterar_pliegues,
                   promedio_por_paciente, umbral_interno, metricas)
RESULTADOS.mkdir(exist_ok=True)

data = leer_datos('metadata')

label_encoder = LabelEncoder()
feature_race = label_encoder.fit_transform(data['NIH Racial Category'].fillna("Unknown")).reshape(-1, 1)
target = data[ETIQUETA].astype(int).values          # CAMBIO: binaria was_preterm
pacientes = data['participant_id'].values
print(data.shape, '| muestras pretérmino:', target.mean().round(3))

In [ ]:
pliegues = cargar_pliegues()
filas, predicciones = [], []
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes, list(ids_tr)), np.isin(pacientes, list(ids_te))
    def ajustar_predecir(X_tr, y_tr, X_va):
        return LogisticRegression(max_iter=200).fit(X_tr, y_tr).predict_proba(X_va)[:, 1]
    # Umbral con datos de entrenamiento solamente (punto iii)
    umbral = umbral_interno(ajustar_predecir, feature_race[tr], target[tr], pacientes[tr],
                            semilla=100 + 10 * rep + fold)
    model = LogisticRegression(max_iter=200)
    model.fit(feature_race[tr], target[tr])
    prob_muestra = model.predict_proba(feature_race[te])[:, 1]
    pp = promedio_por_paciente(pacientes[te], target[te], prob_muestra)
    filas.append(dict(modelo='Regresión logística', rep=rep, fold=fold, **metricas(pp.y, pp.prob, umbral)))
    predicciones.append(pp.assign(rep=rep, fold=fold))

pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A1.csv', index=False)
res = pd.DataFrame(filas)
res.to_csv(RESULTADOS / 'metricas_A1.csv', index=False)
res[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3)